# overview

- This dataset has a "quasi" event log structure, its structure looks like event log, but actually the structure is an artifact of the data extraction process. So for analyzers looking at this dataset, it is important to understand the structure of the dataset.
- Use `dd-parser-cleaner` package notebook utils to load the clean dataset
- Use `featurization.get_package_info()` to inspect interfaces exposed by the package
- Keep `assignment_group == '?'` records through survival dataset preparation, then drop them afterward
- Find the last closed ticket timestamp in the dataset and set it as the observation end
- Set the observation start to one year before the end
- Select tickets opened and closed between the observation start and end as study candidates
- Filter those candidates to closed tickets, group by `assignment_group`, and count unique ticket numbers
- Define `MIN_TICKET_CLOSURE = 20` and keep groups meeting that threshold as `support_level_clear_list`
- Filter the candidate dataset to include only tickets from `support_level_clear_list`
- Run the featurization survival pipeline on the filtered dataset
- Drop records with `assignment_group == '?'` after survival dataset preparation
- Write the prepared survival dataset to the featurization complete dataset location using `featurization` notebook utils and configuration

In [1]:
import inspect
import sys
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "notebooks":
    repo_root = cwd.parent
else:
    repo_root = cwd
sys.path.insert(0, str(repo_root))

import pandas as pd
from featurization import get_package_info, notebook_utils as feat_notebook_utils
from featurization_scripts.featurization import ticket_survival_summary

if (repo_root / "featurizer_config.yaml").exists():
    notebook_dir = repo_root / "notebooks"
elif (repo_root / "notebooks" / "featurizer_config.yaml").exists():
    notebook_dir = repo_root / "notebooks"
else:
    raise FileNotFoundError(
        "Could not locate featurizer_config.yaml in the workspace root."
    )

resolver = feat_notebook_utils.build_notebook_resolver(str(notebook_dir))
raw_clean_dataset = feat_notebook_utils.load_featurization_input_dataset(resolver)
clean_df = raw_clean_dataset.copy()

package_info = get_package_info()
interfaces = (
    package_info.get("interfaces")
    if isinstance(package_info, dict)
    else getattr(package_info, "interfaces", None)
)
print("featurization package interfaces:", interfaces)

if "assignment_group" not in clean_df.columns:
    raise KeyError("assignment_group column is required in the clean dataset")

if "opened_at" in clean_df.columns:
    clean_df["opened_at"] = pd.to_datetime(clean_df["opened_at"], errors="coerce")
else:
    raise KeyError("opened_at column is required for observation period filtering")

if "closed_at" in clean_df.columns:
    clean_df["closed_at"] = pd.to_datetime(clean_df["closed_at"], errors="coerce")
else:
    raise KeyError("closed_at column is required for finding closed tickets")

ticket_col = "ticket_number" if "ticket_number" in clean_df.columns else (
    "number" if "number" in clean_df.columns else None
)
if ticket_col is None:
    raise KeyError("ticket number column is required (ticket_number or number)")

closed_df = clean_df.loc[clean_df["closed_at"].notna()].copy()
if closed_df.empty:
    raise ValueError("No closed tickets found in the dataset")

last_closed_ts = closed_df["closed_at"].max()
observation_end = last_closed_ts
observation_start = observation_end - pd.DateOffset(years=1)

candidate_df = clean_df.loc[
    (clean_df["opened_at"] >= observation_start)
    & (clean_df["closed_at"] <= observation_end)
].copy()

closed_candidates_df = candidate_df.loc[candidate_df["closed_at"].notna()].copy()

group_counts = (
    closed_candidates_df.groupby("assignment_group")[ticket_col]
    .nunique()
    .reset_index(name="ticket_count")
)

MIN_TICKET_CLOSURE = 20
support_level_clear_list = group_counts.loc[
    group_counts["ticket_count"] >= MIN_TICKET_CLOSURE, "assignment_group"
].tolist()

final_df = candidate_df.loc[
    candidate_df["assignment_group"].isin(support_level_clear_list)
].copy()

prepared_survival_df = ticket_survival_summary(
    {"data": final_df, "resolver": resolver},
    stage_cfg={},
)
if not isinstance(prepared_survival_df, pd.DataFrame):
    raise RuntimeError("ticket_survival_summary did not return a DataFrame")

print("Prepared survival DataFrame shape:", prepared_survival_df.shape)
pre_drop_count = int((prepared_survival_df["assignment_group"] == "?").sum())
prepared_survival_df = prepared_survival_df.loc[
    prepared_survival_df["assignment_group"] != "?",
].copy()
print("Prepared survival DataFrame shape after dropping '?':", prepared_survival_df.shape)
output_path = Path(resolver.model_ready_dataset_path)
output_path.parent.mkdir(parents=True, exist_ok=True)
prepared_survival_df.to_csv(output_path, index=False)
print("Saved prepared survival dataset to", output_path)
print("The survival pipeline writes KM and PR data directly to the featurization output directory.")
print(f"Dropped {pre_drop_count} records with assignment_group='?' after survival dataset preparation.")

featurization package interfaces: None
Wrote Kaplan-Meier data (24843 rows, 53 groups) to: /home/rajiv/programming/kmds_migration/itsm_analysis/data/featurization/itsm_KM_data.csv
Wrote parametric regression data (70 rows, 8 groups) to: /home/rajiv/programming/kmds_migration/itsm_analysis/data/featurization/itsm_PR_data.csv
Prepared survival DataFrame shape: (24913, 4)
Prepared survival DataFrame shape after dropping '?': (22755, 4)
Saved prepared survival dataset to /home/rajiv/programming/kmds_migration/itsm_analysis/data/featurization/itsm_survival_model_ready_numeric_data.csv
The survival pipeline writes KM and PR data directly to the featurization output directory.
Dropped 2158 records with assignment_group='?' after survival dataset preparation.


In [2]:
open_count = int((prepared_survival_df["survival_event"] == 0).sum())
closed_count = int((prepared_survival_df["survival_event"] == 1).sum())

print(f"Open tickets in survival dataset: {open_count}")
print(f"Closed tickets in survival dataset: {closed_count}")

Open tickets in survival dataset: 33
Closed tickets in survival dataset: 22722
